In [ ]:
import heapq
import bisect
import datetime
import matplotlib.pyplot as plt

class User:
    """Handles user authentication and budget settings using data structures."""
    def __init__(self, user_id, name, email, password):
        self.user_id = user_id
        self.name = name
        self.email = email
        self.password = password  # In production, use password hashing
        self.budget = 0.0
        self.transactions = []  # Maintains transactions in sorted order (using bisect)
        self.expense_heap = []  # Min-heap to track expenses
    
    def set_budget(self):
        """Prompt user to set a monthly budget limit."""
        amount = float(input("Enter your monthly budget: "))
        self.budget = amount
        print(f"Budget set to ${amount}")

    def add_transaction(self):
        """Prompt user to enter a transaction and maintain sorted order."""
        transaction_id = len(self.transactions) + 1
        amount = float(input("Enter transaction amount: "))
        category = input("Enter transaction category: ")
        t_type = input("Enter transaction type (income/expense): ")
        date = input("Enter transaction date (YYYY-MM-DD): ")
        transaction = Transaction(transaction_id, self.user_id, amount, category, t_type, date)
        bisect.insort(self.transactions, transaction)  # Maintain sorted order
        
        if t_type == 'expense':
            heapq.heappush(self.expense_heap, amount)  # Maintain min-heap of expenses
        
        print(f"Transaction added: {transaction}")

    def get_transactions(self):
        """Returns all transactions sorted by date."""
        return self.transactions
    
    def calculate_expense(self):
        """Calculate total expenses using a heap."""
        return sum(self.expense_heap)

    def check_budget_alert(self):
        """Check if expenses exceed budget using heap data structure."""
        total_expense = self.calculate_expense()
        if total_expense > self.budget:
            print(f"⚠️ Budget exceeded! You have spent ${total_expense}, exceeding your budget of ${self.budget}")

class Transaction:
    """Stores transaction details and supports sorting."""
    def __init__(self, transaction_id, user_id, amount, category, t_type, date):
        self.transaction_id = transaction_id
        self.user_id = user_id
        self.amount = amount
        self.category = category
        self.type = t_type  # "income" or "expense"
        self.date = datetime.datetime.strptime(date, "%Y-%m-%d")
    
    def __lt__(self, other):
        """Comparison method for sorting transactions by date."""
        return self.date < other.date

    def __repr__(self):
        return f"{self.date.date()} - {self.type}: ${self.amount} ({self.category})"

class Report:
    """Generates financial reports."""
    @staticmethod
    def generate_report(user):
        """Generate a sorted financial report."""
        transactions = user.get_transactions()
        print("\n📊 Financial Report")
        print(f"User: {user.name}")
        print("Transactions:")
        for t in transactions:
            print(t)
        print(f"Total Expenses: ${user.calculate_expense()}")
        print(f"Remaining Budget: ${user.budget - user.calculate_expense()}")

class Visualization:
    """Generates graphical representations of financial data."""
    @staticmethod
    def generate_chart(user):
        """Creates a bar chart for expense categories."""
        expense_data = {}
        for t in user.transactions:
            if t.type == 'expense':
                expense_data[t.category] = expense_data.get(t.category, 0) + t.amount
        
        if not expense_data:
            print("No expense data to visualize.")
            return
        
        categories = list(expense_data.keys())
        amounts = list(expense_data.values())

        plt.figure(figsize=(8,5))
        plt.bar(categories, amounts, color='skyblue')
        plt.xlabel("Expense Categories")
        plt.ylabel("Amount ($)")
        plt.title(f"Expense Breakdown for {user.name}")
        plt.show()

# Main program
if __name__ == "__main__":
    user_id = 1
    name = input("Enter your name: ")
    email = input("Enter your email: ")
    password = input("Enter your password: ")
    user = User(user_id, name, email, password)
    
    user.set_budget()
    
    while True:
        add_more = input("Do you want to add a transaction? (yes/no): ")
        if add_more.lower() != 'yes':
            break
        user.add_transaction()
    
    Report.generate_report(user)
    user.check_budget_alert()
    Visualization.generate_chart(user)


Enter your name:  muku
Enter your email:  kudzidube96gmail.com
